# 03 · Retrieval-augmented generation

**Session 01 · notebook 3 of 6**

## Objective

Connect the index to the model: retrieve the passages that match the question and supply them as context. See a retrieval failure, three ways to fix it, and why retrieval and generation are evaluated separately.

## The application after this notebook

```
question ──> retriever ──> top-k passages ──┐
                                             ├──> [ instructions + passages + question ] ──> model ──> answer with citations
question ────────────────────────────────────┘
```

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/avsolatorio/ai4data.git"   # replace with the published repository when released
BRANCH = "feat/training-series"
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "pydantic", "scikit-learn", "requests"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook uses recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
llm = w.LLM()
print("Model client:", llm.mode())

Model client: recorded outputs (no WORKSHOP_API_KEY set)


In [2]:
docs = w.load_documents()
chunks = [c for d in docs for c in w.chunk_sections(d)]
retriever = w.Retriever(chunks)
print(len(chunks), "chunks ·", retriever.embedder.backend)

INSTRUCTIONS = """You are a research assistant for development data.
Answer only from the supplied passages. After each figure, cite the passage in the square-bracket form given.
If the passages do not contain what is needed, say so."""


def rag_answer(question, k=4, scenario="unnamed", where=None, recency_weight=0.0):
    hits = retriever.search(question, k=k, where=where, recency_weight=recency_weight)
    user = f"PASSAGES:\n{w.format_context(hits)}\n\nQUESTION: {question}"
    completion = llm.complete([{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": user}], scenario=scenario)
    return completion, hits

26 chunks · tfidf


## 1. The worked example through the pipeline

Retrieve, then answer. The hits are printed first so that the evidence the model saw is visible.

In [3]:
answer, hits = rag_answer(w.MAIN_QUESTION, k=5, scenario="nb03_rag_main")
w.print_hits(hits)
print()
w.show(answer)

 0.133  [internet-indicator · intro · 2026]
 0.130  [rwanda-digital · Internet use · 2026]
 0.111  [regional-2016 · intro · 2016]
 0.109  [mobile-money · intro · 2026]
 0.105  [kenya-digital · Reading the two indicators together · 2026]

[recorded output: nb03_rag_main]
Internet use. Kenya's share of individuals using the internet reached 35.0 percent of the population
in 2024, up from 7.2 percent in 2010 [kenya-digital · Internet use · 2026]. Rwanda's reached 31.7
percent in 2024, up from 8.0 percent in 2010 [rwanda-digital · Internet use · 2026].

Economic development. Kenya's GDP per capita in current US dollars was 2,133 in 2024, after a dip to
1,943 in 2023 [kenya-economy · GDP per capita · 2026]. Rwanda's was 1,060 in 2024, having passed
1,000 in 2022 [rwanda-economy · GDP per capita · 2026].

Patterns. Kenya has higher levels on both indicators. Rwanda has the faster growth on both, with GDP
growth above 7 percent in most years [rwanda-economy · Growth · 2026]. Internet use conv

## 2. A retrieval failure

"What is the latest available figure for internet use in Kenya?" The 2016 summary contains the words "latest available figures" and ranks high. With a word-based backend it ranks first. The model then answers from it, correctly summarizing an outdated passage.

In [4]:
q = "What is the latest available figure for internet use in Kenya?"
answer, hits = rag_answer(q, k=3, scenario="nb03_rag_outdated")
w.print_hits(hits)
print()
w.show(answer)

 0.221  [regional-2016 · intro · 2016]
 0.159  [kenya-digital · Internet use · 2026]
 0.135  [rwanda-digital · Internet use · 2026]

[recorded output: nb03_rag_outdated]
According to the retrieved summary, 16.6 percent of Kenya's population used the internet in 2015, up
from 7.2 percent in 2010, and the summary notes that the latest available figures are for 2015
[regional-2016 · intro · 2016]. On the trends described there, use was expected to reach about 25
percent by 2020.


The answer is faithful to its evidence and wrong for the reader. Nothing in the generation step can fix this; the evidence was selected badly.

## 3. Three fixes, all on the retrieval side

1. **A metadata filter**: exclude documents dated before 2020 for a question about the latest figure.
2. **A recency weight**: add a bonus to the score for newer documents.
3. **Query reformulation**: search for what the passage would say ("internet use Kenya 2024") instead of what the user asked.

In [5]:
print("1. metadata filter (date >= 2020):")
w.print_hits(retriever.search(q, k=3, where=lambda c: c.date >= "2020"))
print("\n2. recency weight:")
w.print_hits(retriever.search(q, k=3, recency_weight=0.3))
print("\n3. reformulated query:")
w.print_hits(retriever.search("Kenya internet use 2024 percent of population", k=3))

1. metadata filter (date >= 2020):
 0.159  [kenya-digital · Internet use · 2026]
 0.135  [rwanda-digital · Internet use · 2026]
 0.132  [kenya-digital · intro · 2026]

2. recency weight:
 0.419  [kenya-digital · Internet use · 2026]
 0.395  [rwanda-digital · Internet use · 2026]
 0.392  [kenya-digital · intro · 2026]

3. reformulated query:
 0.352  [kenya-digital · intro · 2026]
 0.240  [kenya-digital · Population · 2026]
 0.230  [kenya-digital · Internet use · 2026]


In [6]:
answer, hits = rag_answer(q, k=3, scenario="nb03_rag_recency", recency_weight=0.3)
w.show(answer)

[recorded output: nb03_rag_recency]
The latest available figure is for 2024: 35.0 percent of Kenya's population used the internet, up
from 32.1 percent in 2023 [kenya-digital · Internet use · 2026]. The source is the World Development
Indicators series IT.NET.USER.ZS, as reproduced in the brief.


## 4. Retrieval always returns something

A question outside the collection still gets top-k passages, ranked by whatever is least unlike it. Compare a question the collection can answer with one it cannot.

In [7]:
for q2 in ["What is the population of Rwanda?", "What was inflation in Uganda in 2023?"]:
    print(q2)
    w.print_hits(retriever.search(q2, k=3))
    print()

What is the population of Rwanda?
 0.124  [regional-2016 · intro · 2016]
 0.113  [rwanda-digital · Reading the two indicators together · 2026]
 0.109  [rwanda-digital · Internet use · 2026]

What was inflation in Uganda in 2023?
 0.195  [kenya-economy · Growth · 2026]
 0.195  [rwanda-economy · Growth · 2026]
 0.144  [rwanda-digital · Internet use · 2026]



Both questions get three passages with a score. With the word-based backend the out-of-collection question can even score higher, because it shares words such as "2023" and "percent" with the growth sections. The score alone does not say "no evidence". Two things do: a threshold set from examples on the backend in use, which the evaluation in notebook 06 is for, and instructions that tell the model to say when the passages do not answer the question, which notebook 01 introduced. Applications use both.

## 5. Retrieval and generation are evaluated separately

Retrieval can be tested without calling a model: for each evaluation question, did the top-k passages include a passage from the expected document? The cell runs that test for all eight questions at two values of k.

In [8]:
questions = w.load_questions()
print(f"{'id':4s} {'category':18s} {'k=2':6s} {'k=4':6s} expected documents")
for qd in questions:
    r2 = w.retrieval_hit(retriever.search(qd["question"], k=2), qd["expected_documents"])
    r4 = w.retrieval_hit(retriever.search(qd["question"], k=4), qd["expected_documents"])
    print(f"{qd['id']:4s} {qd['category']:18s} {'hit' if r2 else 'miss':6s} {'hit' if r4 else 'miss':6s} {', '.join(qd['expected_documents'])}")

id   category           k=2    k=4    expected documents
q1   main               hit    hit    kenya-digital, rwanda-digital, kenya-economy, rwanda-economy
q2   factual            hit    hit    rwanda-digital
q3   ambiguous          hit    hit    kenya-digital
q4   missing-data       miss   hit    kenya-digital
q5   out-of-scope       hit    hit    coffee
q6   factual            hit    hit    kenya-economy, rwanda-economy
q7   definition         hit    hit    internet-indicator
q8   outdated-evidence  hit    hit    kenya-digital


A miss here is a retrieval problem, fixed by chunking, metadata, ranking, or the query. A wrong answer with a hit is a generation problem, fixed by instructions or a different model. Mixing the two makes both impossible to improve.

## Key points

- Retrieval selects the evidence; the model writes from it. A faithful answer to the wrong evidence is wrong.
- Fixes for retrieval live in metadata, ranking, and query formulation. The model cannot repair a bad selection.
- Retrieval always returns something; a score threshold turns "least unlike" into "no evidence".
- Test retrieval on its own, with expected documents per question.

## Exercise

Change `chunk_sections` to `chunk_fixed` in the first cell and rerun section 5. Count the misses. Then try k=8.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows.
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.